# TechKnowledge — EDA, Treino e Avaliação do Modelo

**Objetivo:** classificar conteúdo técnico (título + texto) em categorias temáticas.

**Algoritmo:** TF-IDF + Regressão Logística (multinomial)

**Entregáveis:** modelo serializado (`pipeline_classificador.pkl`) + métricas (`metrics.json`)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
import os
import json
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv("../data/dataset.csv")
print(f"Shape: {df.shape}")
df.head()

## 3. Análise Exploratória (EDA)

In [ ]:
# Distribuição das classes
class_counts = df["categoria"].value_counts()
print("Distribuição das classes:")
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Contagem por Categoria")
axes[0].set_xlabel("Categoria")
axes[0].set_ylabel("Quantidade")
axes[0].tick_params(axis="x", rotation=45)

class_counts.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporção por Categoria")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas de comprimento do texto
df["comprimento"] = df["texto"].astype(str).str.len()

print("Estatísticas de comprimento do texto:")
print(df["comprimento"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["comprimento"], bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribuição do Comprimento do Texto")
axes[0].set_xlabel("Comprimento (caracteres)")

sns.boxplot(x="categoria", y="comprimento", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("Comprimento por Categoria")
axes[1].set_xlabel("Categoria")
axes[1].set_ylabel("Comprimento (caracteres)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Verificação de dados ausentes
missing = df.isnull().sum()
print("Valores ausentes por coluna:")
print(missing[missing > 0] if missing.any() else "Nenhum valor ausente encontrado.")

In [ ]:
# Verificação de duplicatas
duplicatas = df.duplicated(subset=["texto"]).sum()
print(f"Textos duplicados: {duplicatas}")

## 4. Limpeza e Pré-processamento

In [ ]:
# Carregar configurações
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Configurações carregadas:")
print(f"  Stopwords: {len(config['tfidf']['stop_words'])} palavras portuguesas")
print(f"  ngram_range: {config['tfidf']['ngram_range']}")
print(f"  max_df: {config['tfidf']['max_df']}, min_df: {config['tfidf']['min_df']}")
print(f"  Algoritmo: {config['model']['algorithm']}, C: {config['model']['C']}")

In [ ]:
# Preparar features e target
X = df["texto"].astype(str).fillna("")
y = df["categoria"]

print(f"Features: {X.shape[0]} amostras")
print(f"Classes: {y.nunique()} categorias")
print(f"Categorias: {sorted(y.unique())}")

## 5. Divisão Treino-Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {len(X_train)} amostras")
print(f"Teste:  {len(X_test)} amostras")
print("\nDistribuição treino:")
print(y_train.value_counts().sort_index())

## 6. Pipeline TF-IDF + Regressão Logística

In [ ]:
min_samples = min(y_train.value_counts())
cv_folds = min(5, min_samples)
print(f"Folds para validação cruzada: {cv_folds}")

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=config['tfidf']['stop_words'],
        ngram_range=tuple(config['tfidf']['ngram_range']),
        max_df=config['tfidf']['max_df'],
        min_df=config['tfidf']['min_df']
    )),
    ('clf', LogisticRegression(
        C=config['model']['C'],
        class_weight='balanced',
        random_state=42,
        max_iter=1000,
        solver='lbfgs'
    ))
])

param_grid = {
    'tfidf__max_df': [0.7, 0.9],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'clf__C': [0.1, 1.0, 10.0]
}

grid = GridSearchCV(pipeline, param_grid, cv=cv_folds, scoring='f1_weighted', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Melhores parâmetros: {grid.best_params_}")

## 7. Avaliação

In [ ]:
y_pred = grid.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"\nAcurácia: {grid.score(X_test, y_test):.4f}")

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=grid.classes_
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de Confusão")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Métricas por classe
metricas_df = pd.DataFrame(report).transpose()
metricas_df

## 8. Teste com Exemplos Reais

In [ ]:
exemplos = [
    "Introdução ao Spring Boot Neste conteúdo são apresentados os conceitos básicos do Spring Boot para criação de APIs REST em Java",
    "Como criar componentes reutilizáveis em React Aprenda a criar componentes funcionais e custom Hooks para reutilizar lógica",
    "Treinando um modelo de classificação com Scikit-Learn Guia prático de como usar TF-IDF e Regressão Logística para classificar textos"
]

for texto in exemplos:
    pred = grid.predict([texto])[0]
    probs = grid.predict_proba([texto])[0]
    idx = list(grid.classes_).index(pred)
    print(f"Texto: {texto[:60]}...")
    print(f"  → Categoria: {pred} (prob: {probs[idx]:.4f})")
    print()

## 9. Serialização

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(grid.best_estimator_, "../models/pipeline_classificador.pkl")
print("✅ Modelo salvo em 'models/pipeline_classificador.pkl'")

In [ ]:
with open("../models/metrics.json", "w") as f:
    json.dump(report, f, indent=4)
print("✅ Métricas salvas em 'models/metrics.json'")

## 10. Extração de Palavras-chave

A extração de palavras-chave reaproveita os pesos TF-IDF + coeficientes do modelo para identificar os termos mais relevantes do texto.

In [ ]:
def extract_keywords(text, pipeline_model, top_n=5):
    tfidf = pipeline_model.named_steps['tfidf']
    clf = pipeline_model.named_steps['clf']
    vec = tfidf.transform([text])
    scores = clf.decision_function(vec)
    class_idx = np.argmax(scores)
    coef = clf.coef_[class_idx]
    features = tfidf.get_feature_names_out()
    contribs = {}
    for idx, val in zip(vec.indices, vec.data):
        if idx < len(coef):
            contribs[features[idx]] = abs(val * coef[idx])
    return [w for w, _ in sorted(contribs.items(), key=lambda x: x[1], reverse=True)[:top_n]]

for texto in exemplos:
    kw = extract_keywords(texto, grid.best_estimator_)
    print(f"Palavras-chave: {kw}")

## Conclusão

O pipeline TF-IDF + Regressão Logística foi treinado e serializado com sucesso.
O modelo está pronto para ser carregado pelo serviço FastAPI em `app/main.py`.